In [2]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
!which python


2.6.0+cu124
False
/home/rebal/anaconda3/envs/KE-CW2/bin/python


In [3]:
!pip install transformers rdflib
!pip install pandas langdetect deepl transformers rdflib python-dotenv torch


In [4]:
import json
import re
import pandas as pd

def clean_lyrics(text, min_word_count=50, max_slashes=10, max_date_mentions=20):
    """
    Cleans a lyrics string by:
    1. Discarding entries that contain "lyrics not found" or "songs chart".
    2. Removing all text before the first '[' (to remove introductory metadata).
    3. Removing any text enclosed in square brackets (e.g., [Chorus: ...], [Verse 1], etc.).
    4. Removing common metadata phrases (e.g., "contributors", "album release", "calendar", "translations").
    5. Removing any occurrence of '//' that is immediately followed by non-whitespace characters (e.g., "//Verse").
    6. Normalizing whitespace.
    7. Filtering out entries that don't reach a minimum word count.
    8. Flagging entries with more than a specified number of "/" characters.
    9. Flagging entries if there are more than max_date_mentions full date occurrences (e.g., "January 3rd 2019").
    """
    lower_text = text.lower()
    # 1. Discard texts that contain unwanted phrases.
    if "lyrics not found" in lower_text or "songs chart" in lower_text:
        return None

    # 2. Remove everything before the first '[' if it exists.
    first_bracket_index = text.find('[')
    if first_bracket_index != -1:
        text = text[first_bracket_index:]
        
    # 3. Remove all content inside square brackets (using DOTALL for multiline).
    text = re.sub(r"\[.*?\]", "", text, flags=re.DOTALL)
    
    # 4. Remove common metadata phrases (case-insensitive).
    metadata_phrases = ["contributors", "album release", "calendar", "translations"]
    for phrase in metadata_phrases:
        text = re.sub(phrase, "", text, flags=re.IGNORECASE)
    
    # 5. Remove any occurrence of '//' attached to a word (e.g., "//Verse").
    text = re.sub(r"//\S+", "", text)
    
    # 6. Normalize whitespace: collapse multiple newlines and trim extra spaces.
    text = re.sub(r"\n+", "\n", text).strip()
    
    # 7. Check if cleaned text meets a minimum word count.
    if len(text.split()) < min_word_count:
        return None

    # 8. Check if the cleaned text contains too many "/" characters.
    if text.count("/") > max_slashes:
        return None

    # 9. Check for full date patterns and flag if there are more than max_date_mentions.
    # This regex matches a month name, a space, one or two digits with optional st/nd/rd/th, space, and 4-digit year.
    date_matches = re.findall(r"[A-Za-z]+ \d{1,2}(?:st|nd|rd|th)?[,]? \d{4}", text)
    if len(date_matches) > max_date_mentions:
        return None

    return text

# Load the JSON file containing lyrics (assumed filename "lyrics.json")
with open("lyrics.json", "r", encoding="utf-8") as f:
    lyrics_data = json.load(f)

# Prepare lists for valid (cleaned) and flagged entries.
cleaned_entries = []
flagged_entries = []

# Process each entry in the JSON file.
for key, lyrics in lyrics_data.items():
    cleaned = clean_lyrics(lyrics)
    
    # Split the key into artist and title using " - " as separator.
    if " - " in key:
        artist, title = key.split(" - ", 1)
    else:
        artist = key
        title = ""
    
    if cleaned is None:
        flagged_entries.append({
            "artist": artist,
            "title": title,
            "original_lyrics": lyrics
        })
    else:
        cleaned_entries.append({
            "artist": artist,
            "title": title,
            "cleaned_lyrics": cleaned
        })

# Convert the cleaned and flagged entries to DataFrames.
df_cleaned = pd.DataFrame(cleaned_entries)
df_flagged = pd.DataFrame(flagged_entries)

# Save the cleaned entries to a CSV file.
df_cleaned.to_csv("lyrics_cleaned.csv", index=False, encoding="utf-8")
print("Cleaned lyrics have been saved to 'lyrics_cleaned.csv'.")

# Save the flagged entries to a JSON file for analysis.
with open("lyrics_flagged.json", "w", encoding="utf-8") as f:
    json.dump(flagged_entries, f, indent=4, ensure_ascii=False)
print("Flagged entries have been saved to 'lyrics_flagged.json'.")

# Optionally, display the first few cleaned entries.
print(df_cleaned.head())


Cleaned lyrics have been saved to 'lyrics_cleaned.csv'.
Flagged entries have been saved to 'lyrics_flagged.json'.
                     artist                 title  \
0      Usherfeat. will.i.am                   OMG   
1  Katy Perryft. Snoop Dogg      California Gurls   
2  B.o.Bft. Hayley Williams             Airplanes   
3                     Ke$ha  Your Love Is My Drug   
4     Taio Cruzft. Ludacris      Break Your Heart   

                                      cleaned_lyrics  
0  Oh my gosh\nBaby, let me\nI did it again, so I...  
1  Greetings, loved ones\nLet's take a journey\nI...  
2  Can we pretend that\nAirplanes in the night sk...  
3  1 Contributor\nAll Hands On Deck! - Your Love ...  
4  Woah\nWoah\nWoah\nWoah (Uh)\nNow listen to me,...  




## Lyrics Theme Extraction Pipeline
# 
 This notebook will:
 1. Load a CSV database of song lyrics.
 2. Use DeepL to detect the language and translate non-English lyrics into English.
 3. Use Bert-Uncased emotion’s sentiment analysis API to extract the theme from the (translated) lyrics.
 4. Save the enriched data (with `hasLanguage` and `hasTheme`) to a new CSV file.


In [7]:
import pandas as pd
import os
from dotenv import load_dotenv
import deepl  
from langdetect import detect
import requests
from transformers import pipeline
from rdflib import Graph, URIRef, Literal, Namespace
import deepl
from collections import Counter
load_dotenv()
DEEPL_AUTH_KEY = os.getenv("DEEPL_AUTH_KEY")



In [8]:
auth_key = DEEPL_AUTH_KEY
translator = deepl.Translator(auth_key)



# Function to Translate Lyrics Using DeepL

 DeepL automatically detects the source language and returns both the detected language and the translated text.


In [26]:

def translate_lyrics(text):
    """
    Uses the first 80 characters for detection.
    If not English, translates the full text.
    """
    snippet = text[:80]
    try:
        detect_result = translator.translate_text(snippet, target_lang="EN-US")
        detected_lang = detect_result.detected_source_lang
    except Exception as e:
        print("❌ Detection error:", e)
        return "unknown", text

    if detected_lang.upper() == "EN":
        return detected_lang, ""  # No need to store translation
    else:
        try:
            full_result = translator.translate_text(text, target_lang="EN-US")
            return full_result.detected_source_lang, full_result.text
        except Exception as e:
            print("❌ Translation error:", e)
            return "unknown", text



# Function to Extract Song Theme Using Distilbert’s Sentiment Analysis A




In [11]:
from transformers import pipeline

emotion_classifier = pipeline(
    "text-classification",
    model="bhadresh-savani/distilbert-base-uncased-emotion",
    framework="pt"    
)


def extract_theme(text):
    try:
        # Get predictions from the model
        results = emotion_classifier(text)
        if results:
            # Pick the label with the highest score
            top = max(results, key=lambda x: x['score'])
            return top['label']
        else:
            return "unknown"
    except Exception as e:
        print("Theme extraction error:", e)
        return "unknown"

Device set to use cpu


In [10]:


def extract_theme(text):
    try:
        # Get predictions from the model
        results = emotion_classifier(text)
        if results:
            # Pick the label with the highest score
            top = max(results, key=lambda x: x['score'])
            return top['label']
        else:
            return "unknown"
    except Exception as e:
        print("Theme extraction error:", e)
        return "unknown"

# Load the Lyrics Database
 
 Assume your CSV file `lyrics.csv` has at least the following columns:
 - `id`
 - `title`
 - `lyrics`


In [11]:
df = pd.read_csv("lyrics_cleaned.csv")
print(f"Loaded {len(df)} lyric entries.")


Loaded 2144 lyric entries.


In [12]:
import pandas as pd

df = pd.read_csv("lyrics_cleaned.csv")

# Total characters in the "cleaned_lyrics" column
total_chars = df["cleaned_lyrics"].astype(str).str.len().sum()
print(f"Total characters in 'cleaned_lyrics': {total_chars}")
print("Number of lyrics:", len(df))

Total characters in 'cleaned_lyrics': 5817058
Number of lyrics: 2144


 ### Process Each Lyric Entry

 For each entry, we:
 1. Use DeepL to detect language and translate lyrics (if necessary).
 2. Use Model to extract the song theme.
 3. Store the detected language, translated lyrics, and theme.


In [18]:
# Create new columns in the dataframe
df["detected_language"] = None
df["translated_lyrics"] = None
df["theme"] = None


# Process each lyric for language detection + translation
for idx, row in df.iterrows():
    lyrics = row["cleaned_lyrics"]
    detected_lang, translated = translate_lyrics(lyrics)
    df.at[idx, "detected_language"] = detected_lang
    df.at[idx, "translated_lyrics"] = translated
    
    if idx % 50 == 0 and idx != 0:
        print(f"✅ Processed {idx} lyrics so far")
        print(f"Example {idx}:")
        print(f" - Language: {detected_lang}")
        print(f" - Original snippet: {lyrics[:60]}...")
        print(f" - Translated snippet: {translated[:60]}...\n")
# Save intermediate CSV
df.to_csv("lyrics_with_language.csv", index=False, encoding="utf-8")
print(" Saved to 'lyrics_with_language.csv'")



✅ Processed 50 lyrics so far
Example 50:
 - Language: EN
 - Original snippet: La-la-la-la
La-la-la-la
La-la-la-la (Uh, yeah)
La-la-la-la
I...
 - Translated snippet: La-la-la-la
La-la-la-la
La-la-la-la (Uh, yeah)
La-la-la-la
I...

✅ Processed 100 lyrics so far
Example 100:
 - Language: EN
 - Original snippet: I remember saying I don't care either way
Just as long as he...
 - Translated snippet: I remember saying I don't care either way
Just as long as he...

✅ Processed 150 lyrics so far
Example 150:
 - Language: EN
 - Original snippet: There's a place downtown where the freaks all come around
It...
 - Translated snippet: There's a place downtown where the freaks all come around
It...

✅ Processed 200 lyrics so far
Example 200:
 - Language: EN
 - Original snippet: When the roll is called up yonder
I hope you see me there
It...
 - Translated snippet: When the roll is called up yonder
I hope you see me there
It...

✅ Processed 250 lyrics so far
Example 250:
 - Language: EN
 - Original sni

In [ ]:
from transformers import pipeline
import pandas as pd
from collections import Counter

# Load lyrics_with_language_updated.csv (must contain columns: 
# "cleaned_lyrics", "detected_language", "translated_lyrics")
df = pd.read_csv("lyrics_with_language_updated.csv")

# Load transformer model for emotion classification.
# This model will run locally using PyTorch.
emotion_classifier = pipeline(
    "text-classification",
    model="bhadresh-savani/distilbert-base-uncased-emotion",
    framework="pt"
)

def extract_theme(text, chunk_size=1000, max_chunks=3):
    """
    Splits the input text into chunks of up to `chunk_size` characters,
    but processes only the first `max_chunks` chunks.
    Runs the emotion classifier on each chunk and returns the dominant emotion.
    """
    try:
        # Split the text into chunks
        chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]
        # Only process the first max_chunks chunks
        chunks = chunks[:max_chunks]
        
        emotions = []
        for chunk in chunks:
            results = emotion_classifier(chunk)
            if results:
                # Choose the label with the highest score for this chunk
                top_result = max(results, key=lambda x: x['score'])
                emotions.append(top_result['label'])
        
        # Return the most common emotion (dominant emotion)
        if emotions:
            dominant_emotion = Counter(emotions).most_common(1)[0][0]
            return dominant_emotion
        else:
            return "unknown"
    except Exception as e:
        print("Theme extraction error:", e)
        return "unknown"

# Create a new 'theme' column
df["theme"] = ""

# Process each row for theme extraction:
for idx, row in df.iterrows():
    detected_lang = str(row["detected_language"]).upper()
    
    # If detected language is UNKNOWN, skip theme extraction.
    if detected_lang == "UNKNOWN":
        df.at[idx, "theme"] = ""
    else:
        # If language is English, use original cleaned lyrics,
        # otherwise use the translated lyrics.
        text_for_analysis = row["cleaned_lyrics"] if detected_lang == "EN" else row["translated_lyrics"]
        theme = extract_theme(text_for_analysis)
        df.at[idx, "theme"] = theme
    
    # Checkpoint: every 100 rows, save the DataFrame to file.
    if (idx + 1) % 100 == 0:
        df.to_csv("lyrics_enriched.csv", index=False, encoding="utf-8")
        print(f"Checkpoint: Processed {idx + 1} entries saved to 'lyrics_enriched.csv'.")
    
    if idx % 2 == 0 and idx != 0:
        print(f"Processed {idx} entries; example theme: {theme}")
# Final save
df.to_csv("lyrics_enriched.csv", index=False, encoding="utf-8")
print("✅ Enriched CSV with themes saved to 'lyrics_enriched.csv'")


Device set to use cpu


Processed 2 entries; example theme: joy
Processed 4 entries; example theme: anger
Processed 6 entries; example theme: joy
Processed 8 entries; example theme: joy
Processed 10 entries; example theme: joy
Processed 12 entries; example theme: sadness
Processed 14 entries; example theme: love
Processed 16 entries; example theme: sadness
Processed 18 entries; example theme: love
Processed 20 entries; example theme: joy
Processed 22 entries; example theme: joy
Processed 24 entries; example theme: joy
Processed 26 entries; example theme: sadness
Processed 28 entries; example theme: anger
Processed 30 entries; example theme: anger
Processed 32 entries; example theme: joy
Processed 34 entries; example theme: joy
Processed 36 entries; example theme: joy
Processed 38 entries; example theme: joy
Processed 40 entries; example theme: joy
Processed 42 entries; example theme: anger
Processed 44 entries; example theme: joy
Processed 46 entries; example theme: joy
Processed 48 entries; example theme: jo

Token indices sequence length is longer than the specified maximum sequence length for this model (525 > 512). Running this sequence through the model will result in indexing errors


Theme extraction error: The size of tensor a (525) must match the size of tensor b (512) at non-singleton dimension 1
Processed 402 entries; example theme: unknown
Processed 404 entries; example theme: sadness
Processed 406 entries; example theme: anger
Processed 408 entries; example theme: anger
Processed 410 entries; example theme: joy
Processed 412 entries; example theme: sadness
Processed 414 entries; example theme: love
Processed 416 entries; example theme: joy
Processed 418 entries; example theme: anger
Processed 420 entries; example theme: joy
Processed 422 entries; example theme: sadness
Processed 424 entries; example theme: joy
Processed 426 entries; example theme: joy
Processed 428 entries; example theme: joy
Processed 430 entries; example theme: anger
Processed 432 entries; example theme: joy
Processed 434 entries; example theme: joy
Processed 436 entries; example theme: surprise
Processed 438 entries; example theme: joy
Processed 440 entries; example theme: joy
Processed 44

In [12]:
import pandas as pd
from transformers import pipeline
from collections import Counter

# Load the current enriched CSV file that contains rows 0 to 1399 already processed.
df = pd.read_csv("lyrics_enriched.csv")
print(f"Loaded {len(df)} entries from 'lyrics_enriched.csv'.")

# Initialize the emotion classifier for theme extraction.
emotion_classifier = pipeline(
    "text-classification",
    model="bhadresh-savani/distilbert-base-uncased-emotion",
    framework="pt"
)

def extract_theme(text, chunk_size=1000, max_chunks=3):
    """
    Splits the input text into chunks of up to `chunk_size` characters,
    processes only the first `max_chunks` chunks, and returns the dominant emotion.
    """
    try:
        # Split text into chunks
        chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]
        # Only process the first max_chunks chunks.
        chunks = chunks[:max_chunks]
        emotions = []
        for chunk in chunks:
            results = emotion_classifier(chunk)
            if results:
                top_result = max(results, key=lambda x: x['score'])
                emotions.append(top_result['label'])
        if emotions:
            return Counter(emotions).most_common(1)[0][0]
        else:
            return "unknown"
    except Exception as e:
        print("Theme extraction error:", e)
        return "unknown"

# Resume processing from row 1401 (index 1400 in 0-indexing)
start_index = 1400
print(f"Resuming processing from entry {start_index + 1}...")

# Process remaining rows
for idx, row in df.iloc[start_index:].iterrows():
    detected_lang = str(row["detected_language"]).upper()  # e.g. 'EN', 'ES', etc.
    
    # If detected language is UNKNOWN, skip theme extraction.
    if detected_lang == "UNKNOWN":
        df.at[idx, "theme"] = ""
    else:
        # If English, use the original cleaned lyrics; otherwise, use the translated lyrics.
        text_for_analysis = row["cleaned_lyrics"] if detected_lang == "EN" else row["translated_lyrics"]
        theme = extract_theme(text_for_analysis)
        df.at[idx, "theme"] = theme

    # Save a checkpoint every 100 processed entries from the resumed set.
    if (idx + 1 - start_index) % 100 == 0:
        df.to_csv("lyrics_enriched.csv", index=False, encoding="utf-8")
        print(f"Checkpoint: Processed {idx + 1} entries (resumed portion) saved.")
    if idx % 2 == 0 and idx != 0:
        print(f"Processed {idx} entries; example theme: {theme}")
# Final save after processing all remaining entries.
df.to_csv("lyrics_enriched.csv", index=False, encoding="utf-8")
print("Processing complete. Updated CSV saved to 'lyrics_enriched.csv'.")


Loaded 2144 entries from 'lyrics_enriched.csv'.


Device set to use cpu


Resuming processing from entry 1401...
Processed 1400 entries; example theme: anger
Processed 1402 entries; example theme: anger
Processed 1404 entries; example theme: anger
Processed 1406 entries; example theme: sadness
Processed 1408 entries; example theme: joy
Processed 1410 entries; example theme: joy
Processed 1412 entries; example theme: fear
Processed 1414 entries; example theme: anger
Processed 1416 entries; example theme: joy
Processed 1418 entries; example theme: sadness
Processed 1420 entries; example theme: joy
Processed 1422 entries; example theme: anger
Processed 1424 entries; example theme: anger
Processed 1426 entries; example theme: anger
Processed 1428 entries; example theme: joy
Processed 1430 entries; example theme: anger
Processed 1432 entries; example theme: joy
Processed 1434 entries; example theme: anger
Processed 1436 entries; example theme: sadness
Processed 1438 entries; example theme: love
Processed 1440 entries; example theme: joy
Processed 1442 entries; ex

Token indices sequence length is longer than the specified maximum sequence length for this model (791 > 512). Running this sequence through the model will result in indexing errors


Theme extraction error: The size of tensor a (791) must match the size of tensor b (512) at non-singleton dimension 1
Processed 1592 entries; example theme: unknown
Processed 1594 entries; example theme: anger
Processed 1596 entries; example theme: joy
Processed 1598 entries; example theme: joy
Checkpoint: Processed 1600 entries (resumed portion) saved.
Processed 1600 entries; example theme: joy
Processed 1602 entries; example theme: anger
Processed 1604 entries; example theme: anger
Processed 1606 entries; example theme: joy
Processed 1608 entries; example theme: fear
Processed 1610 entries; example theme: joy
Processed 1612 entries; example theme: joy
Processed 1614 entries; example theme: joy
Processed 1616 entries; example theme: sadness
Processed 1618 entries; example theme: anger
Processed 1620 entries; example theme: sadness
Processed 1622 entries; example theme: anger
Processed 1624 entries; example theme: joy
Processed 1626 entries; example theme: joy
Processed 1628 entries; e

In [13]:
import pandas as pd

# Load the enriched lyrics CSV
df = pd.read_csv("lyrics_enriched.csv")

# Extract only the 'theme' column
theme_only = df[["theme"]]

# Save to a new CSV file
theme_only.to_csv("lyrics_themes_only.csv", index=False, encoding="utf-8")

print(" Saved themes to 'lyrics_themes_only.csv'")


✅ Saved themes to 'lyrics_themes_only.csv'
